# Explore a big dataframe: NYC yellow-taxi trips

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/nyc-taxi-10m/nyc_taxi_10m.ipynb)

Over twelve million real taxi trips in one scrolling grid, with a histogram in every column header, a profile panel, filters and sort that run over **all** the rows, and the rows you pick coming back to Python as a pandas `DataFrame`. No API key.

**Data:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), yellow-taxi trips, January to April 2024 (12,569,245 trips after cleaning) with obvious junk rows (bad dates, 0-mile or day-long trips) dropped, so the header histograms show real shapes, plus the TLC taxi-zone lookup. Published by the NYC Taxi & Limousine Commission as open data, free to use; fetched from the TLC's public CDN when you run the notebook (about 210 MB). The notebook code is MIT-licensed (see the repo `LICENSE`); the data stays under the TLC's terms.

**Memory:** the 12.6M-row frame takes 0.9 GB in memory, which fits a free Colab runtime (12 GB).

In [1]:
%pip install -q "lattice-grid-jupyter[large]>=1.111.4.0"

## 1. Fetch the trips

In [2]:
import os, urllib.request, duckdb
from lattice_grid_jupyter import LatticeGridWidget

CDN = "https://d37ci6vzurychx.cloudfront.net/"
files = [f"yellow_tripdata_2024-{m:02d}.parquet" for m in (1, 2, 3, 4)]
for name in files + ["taxi_zone_lookup.csv"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(CDN + ("misc/" if name.endswith("csv") else "trip-data/") + name, name)

trips = duckdb.sql(f"""
    SELECT tpep_pickup_datetime AS pickup, passenger_count AS passengers, trip_distance AS miles,
           date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime) AS minutes,
           pu.Zone AS pickup_zone, dz.Zone AS dropoff_zone,
           payment_type, fare_amount AS fare, tip_amount AS tip, total_amount AS total
    FROM read_parquet({files}) t
    LEFT JOIN 'taxi_zone_lookup.csv' pu ON t.PULocationID = pu.LocationID
    LEFT JOIN 'taxi_zone_lookup.csv' dz ON t.DOLocationID = dz.LocationID
    WHERE tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-05-01'
      AND trip_distance BETWEEN 0.1 AND 100 AND fare_amount BETWEEN 1 AND 500
      AND date_diff('minute', tpep_pickup_datetime, tpep_dropoff_datetime) BETWEEN 1 AND 240
    ORDER BY tpep_pickup_datetime
    """).df()
for col in ("pickup_zone", "dropoff_zone"):
    trips[col] = trips[col].astype("category")
print(f"{len(trips):,} trips, {trips.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

12,569,245 trips, 0.9 GB in memory


## 2. Open them in a grid

Above 100,000 rows the widget switches to its **large-dataframe mode** on its own: no rows are sent up front, the grid asks Python for each window as you scroll, and sort, filter, histograms and the profile are computed over every row. `histograms=True` puts a distribution in each header; `profile=True` adds a **Statistics** tab on the right edge.

The grid starts filtered to the trips we want: **more than 10 miles, dropped off at an airport**. Try the header histograms (click a bar to filter), sort a column, or use the filter menu.

In [3]:
airport_trips = {"op": "and", "conditions": [
    {"col": "dropoff_zone", "op": "contains", "value": "Airport"},
    {"col": "miles", "op": "gt", "value": 10}]}
widths = dict(pickup=160, passengers=120, miles=150, minutes=110, pickup_zone=190, dropoff_zone=190,
              payment_type=140, fare=90, tip=90, total=100)
w = LatticeGridWidget(trips, histograms=True, profile=True, height=560,
                      columns=[{"field": f, "layout": {"width": x}} for f, x in widths.items()],
                      state={"filters": airport_trips})
w

![the grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/nyc-taxi-10m/grid.png)

*The grid in JupyterLab: all ten columns, a histogram strip in every header, and the airport filter applied. The strips are computed over all 12.6M rows, so skewed columns such as miles and fare look like a spike at the low end.*

## 3. Pick rows, get a DataFrame

Click a row, shift-click to extend, or Ctrl/Cmd-click to add more. Then run the next cell: `w.selected` is the selection as an ordinary pandas `DataFrame` with the original index and dtypes. If you have not picked anything, it falls back to the first 5,000 rows of the filtered view (`w.view`, a lazy handle in this mode), so the notebook also runs top to bottom unattended.

In [4]:
sel = w.selected
if sel.empty:
    sel = w.view.head(5000)
print(f"{len(w.view):,} airport trips over 10 miles in the grid; {len(sel):,} rows in the DataFrame")
sel.groupby("dropoff_zone", observed=True).agg(trips=("fare", "size"), avg_miles=("miles", "mean"),
    avg_minutes=("minutes", "mean"), avg_fare=("fare", "mean"), avg_tip=("tip", "mean")).round(2)

213,025 airport trips over 10 miles in the grid; 5,000 rows in the DataFrame


,trips,avg_miles,avg_minutes,avg_fare,avg_tip
dropoff_zone,,,,,
JFK Airport,2940,17.69,47.26,69.80,9.36
LaGuardia Airport,1197,11.64,27.02,47.99,8.03
Newark Airport,863,18.65,35.12,94.00,11.79


## 4. Pivot in pandas

The grid's own pivot mode is not pushed down to Python in the large-dataframe mode (the grid refuses it by name), so pivot with pandas on a slice of the view: `w.view.head(n)` is the first `n` rows of whatever the grid is currently filtered and sorted to.

In [5]:
w.view.head(20000).pivot_table(index="pickup_zone", columns="dropoff_zone", values="fare",
                                  aggfunc="mean", observed=True).round(1).dropna(how="all").head(10)

dropoff_zone,JFK Airport,LaGuardia Airport,Newark Airport
pickup_zone,,,
Alphabet City,70.0,38.1,NaN
Arrochar/Fort Wadsworth,NaN,NaN,76.9
Astoria,56.9,NaN,91.5
Battery Park,70.0,56.9,NaN
Battery Park City,71.7,59.4,80.2
Bay Ridge,65.2,67.8,NaN
Bayside,36.9,NaN,NaN
Bedford,49.7,NaN,NaN
Bensonhurst East,61.8,NaN,NaN
